# 🧠 Emotional Distress Detection Analytics Pipeline

This notebook demonstrates the NLP analytics pipeline for detecting early emotional distress signals from Instagram data.

## System Overview

The pipeline consists of two stages:

1. **Stage 1: NLP Signal Extraction** - Analyzes text (captions/comments) for:
   - Sentiment (positive/negative/neutral)
   - Emotions (sadness, anger, fear, joy, etc.)
   - Cognitive distortions (catastrophizing, hopelessness, etc.)

2. **Stage 2: Behavioral Feature Engineering** - Aggregates signals to compute:
   - Distortion metrics and trends
   - Sentiment volatility
   - Engagement patterns
   - Risk scores and priorities

## Setup
First, ensure you're in the backend directory and have the virtual environment activated.

In [1]:
import os
import sys
import asyncio
from datetime import datetime

# Add backend to path
backend_path = os.path.join(os.getcwd(), 'backend')
if backend_path not in sys.path:
    sys.path.insert(0, backend_path)

print(f"✅ Backend path added: {backend_path}")
print(f"📁 Current directory: {os.getcwd()}")

✅ Backend path added: /Users/pandeymahi/Documents/GitHub/DellInnovate2026_Team-Untitled/backend
📁 Current directory: /Users/pandeymahi/Documents/GitHub/DellInnovate2026_Team-Untitled


## Step 1: Test MongoDB Connection

Let's verify the database connection before running the pipeline.

In [2]:
from config.database import MongoDB
from loguru import logger

async def test_connection():
    """Test MongoDB connection"""
    try:
        await MongoDB.connect_db()
        db = MongoDB.get_db()
        
        # Count documents in collections
        posts_count = await db.instagram_posts.count_documents({})
        users_count = await db.instagram_users.count_documents({})
        
        print("=" * 60)
        print("📊 Database Connection Status")
        print("=" * 60)
        print(f"✅ Connected to MongoDB")
        print(f"📝 Instagram posts: {posts_count}")
        print(f"👤 Instagram users: {users_count}")
        print("=" * 60)
        
        return True
    except Exception as e:
        print(f"❌ Connection failed: {e}")
        return False

# Run the test
await test_connection()

2026-02-28 20:50:49.140 | SUCCESS  | config.database:connect_db:26 - Connected to MongoDB database: instagram_scraper


📊 Database Connection Status
✅ Connected to MongoDB
📝 Instagram posts: 164
👤 Instagram users: 11


True

## Step 2: Test Individual NLP Models

Let's test each NLP model independently to ensure they're working correctly.

In [4]:
from analytics.nlp_models import SentimentAnalyzer, EmotionDetector, CognitiveDistortionDetector

# Sample texts for testing
test_texts = [
    "I love this post! So inspiring and beautiful!",  # Positive
    "I always fail at everything. Nothing ever works out.",  # Negative with distortion
    "Feeling really sad today. Nobody understands me.",  # Sadness
    "This is just a regular comment.",  # Neutral
]

print("🧪 Testing NLP Models")
print("=" * 70)

# Test Sentiment Analysis
print("\n📊 Sentiment Analysis:")
print("-" * 70)
sentiment_analyzer = SentimentAnalyzer()

for text in test_texts[:2]:  # Test first 2
    result = sentiment_analyzer.analyze(text)
    print(f"\nText: {text[:60]}...")
    print(f"→ Label: {result['label']} (score: {result['score']:.3f})")
    print(f"→ Sentiment score: {result['sentiment_score']:.3f}")

print("\n" + "=" * 70)

2026-02-28 20:51:34.541 | INFO     | analytics.nlp_models:__init__:33 - Loading sentiment model: cardiffnlp/twitter-roberta-base-sentiment-latest


🧪 Testing NLP Models

📊 Sentiment Analysis:
----------------------------------------------------------------------


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 590.91it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.weight     | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 
roberta.pooler.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-02-28 20:51:37.552 | SUCCESS  | analytics.nlp_models:__init__:45 - Sentiment model loaded on cpu



Text: I love this post! So inspiring and beautiful!...
→ Label: positive (score: 0.985)
→ Sentiment score: 0.979

Text: I always fail at everything. Nothing ever works out....
→ Label: negative (score: 0.891)
→ Sentiment score: -0.871



In [5]:
# Test Emotion Detection
print("\n😊 Emotion Detection:")
print("-" * 70)
emotion_detector = EmotionDetector()

for text in test_texts:
    result = emotion_detector.detect(text)
    print(f"\nText: {text[:60]}...")
    print(f"→ Emotion: {result['label']} (confidence: {result['score']:.3f})")
    print(f"→ Distress: {result['is_distress']} (score: {result['distress_score']:.3f})")

print("\n" + "=" * 70)

2026-02-28 20:51:47.540 | INFO     | analytics.nlp_models:__init__:131 - Loading emotion model: j-hartmann/emotion-english-distilroberta-base



😊 Emotion Detection:
----------------------------------------------------------------------


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 2123.88it/s, Materializing param=roberta.encoder.layer.5.output.dense.weight]             
RobertaForSequenceClassification LOAD REPORT from: j-hartmann/emotion-english-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-02-28 20:51:49.443 | SUCCESS  | analytics.nlp_models:__init__:143 - Emotion model loaded on cpu



Text: I love this post! So inspiring and beautiful!...
→ Emotion: joy (confidence: 0.987)
→ Distress: False (score: 0.003)

Text: I always fail at everything. Nothing ever works out....
→ Emotion: neutral (confidence: 0.329)
→ Distress: False (score: 0.522)

Text: Feeling really sad today. Nobody understands me....
→ Emotion: sadness (confidence: 0.986)
→ Distress: True (score: 0.987)

Text: This is just a regular comment....
→ Emotion: neutral (confidence: 0.965)
→ Distress: False (score: 0.011)



In [6]:
# Test Cognitive Distortion Detection
print("\n🧠 Cognitive Distortion Detection:")
print("-" * 70)
distortion_detector = CognitiveDistortionDetector(threshold=0.55)

for text in test_texts:
    result = distortion_detector.detect(text)
    print(f"\nText: {text[:60]}...")
    print(f"→ Distortion detected: {result['distortion_indicator'] == 1}")
    if result['distortion_category']:
        print(f"→ Type: {result['distortion_category']}")
    print(f"→ Score: {result['distortion_score']:.3f}")

print("\n" + "=" * 70)

2026-02-28 20:52:02.038 | INFO     | analytics.nlp_models:__init__:278 - Loading distortion detection model: sentence-transformers/all-MiniLM-L6-v2



🧠 Cognitive Distortion Detection:
----------------------------------------------------------------------


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1562.81it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-02-28 20:52:19.697 | SUCCESS  | analytics.nlp_models:__init__:289 - Cognitive distortion detector initialized



Text: I love this post! So inspiring and beautiful!...
→ Distortion detected: False
→ Score: 0.183

Text: I always fail at everything. Nothing ever works out....
→ Distortion detected: True
→ Type: overgeneralization
→ Score: 0.801

Text: Feeling really sad today. Nobody understands me....
→ Distortion detected: False
→ Score: 0.346

Text: This is just a regular comment....
→ Distortion detected: False
→ Score: 0.244



## Step 3: Run Stage 1 - NLP Signal Extraction

This stage extracts text from Instagram posts/comments and runs all NLP models.

**Note:** This may take several minutes depending on the number of posts in your database.

In [10]:
# Reload the modules to pick up the latest changes
import importlib
import analytics.text_preprocessing
import analytics.signal_extraction

importlib.reload(analytics.text_preprocessing)
importlib.reload(analytics.signal_extraction)

print("✅ Modules reloaded successfully")

✅ Modules reloaded successfully


In [ ]:
from analytics.signal_extraction import NLPSignalExtractor

async def run_stage1(case_users=None, limit=10):
    """
    Run Stage 1: NLP Signal Extraction
    
    Args:
        case_users: List of specific usernames (None = all)
        limit: Max posts to process (None = all)
    """
    print("🚀 Starting Stage 1: NLP Signal Extraction")
    print("=" * 70)
    
    extractor = NLPSignalExtractor()
    
    results = await extractor.run_pipeline(
        case_users=case_users,
        limit=limit,
        export_csv=True,
        csv_path="nlp_signals_output.csv"
    )
    
    return results

# Run Stage 1 with a small limit for testing
# Change limit=None to process all posts
stage1_results = await run_stage1(limit=10)

print("\n📊 Stage 1 Results:")
print(f"   • Text units found: {stage1_results['text_units_found']}")
print(f"   • Valid units analyzed: {stage1_results['valid_units']}")
print(f"   • Signals created: {stage1_results['signals_created']}")
print(f"   • CSV exported: {stage1_results['csv_file']}")
print(f"   • Duration: {stage1_results['duration_seconds']:.2f}s")

2026-02-28 21:01:06.051 | INFO     | analytics.signal_extraction:__init__:52 - NLP Signal Extractor initialized
2026-02-28 21:01:06.054 | INFO     | analytics.signal_extraction:run_pipeline:292 - ======================================================================
2026-02-28 21:01:06.055 | INFO     | analytics.signal_extraction:run_pipeline:293 - Starting NLP Signal Extraction Pipeline (Stage 1)
2026-02-28 21:01:06.058 | INFO     | analytics.signal_extraction:run_pipeline:294 - ======================================================================
2026-02-28 21:01:06.059 | INFO     | analytics.signal_extraction:run_pipeline:297 - [1/5] Extracting text units from database...
2026-02-28 21:01:06.208 | INFO     | analytics.signal_extraction:extract_text_units_from_db:88 - Retrieved 10 posts from database
2026-02-28 21:01:06.208 | INFO     | analytics.signal_extraction:extract_text_units_from_db:96 - Extracted 94 text units
2026-02-28 21:01:06.208 | INFO     | analytics.signal_extraction

🚀 Starting Stage 1: NLP Signal Extraction


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1922.08it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.weight     | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 
roberta.pooler.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-02-28 21:01:08.900 | SUCCESS  | analytics.nlp_models:__init__:45 - Sentiment model loaded on cpu
2026-02-28 21:01:08.901 | INFO     | analytics.nlp_models:__init__:131 - Loading emotion model: j-hartmann/emotion-english-distilroberta-base
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 1444.51it/s, Materializing param=roberta.encoder.layer.5.output.dense.weigh

## Step 4: Inspect NLP Signals

Let's examine some of the signals we just created.

In [ ]:
async def inspect_signals(limit=5):
    """View sample signals from database"""
    db = MongoDB.get_db()
    signals_collection = db.text_units_signals
    
    signals = await signals_collection.find().limit(limit).to_list(length=None)
    
    print("=" * 70)
    print(f"🔍 Inspecting {len(signals)} Sample Signals")
    print("=" * 70)
    
    for i, signal in enumerate(signals, 1):
        print(f"\n[{i}] Case User: {signal['case_user']}")
        print(f"    Text: {signal['text'][:80]}...")
        print(f"    Sentiment: {signal['sentiment_label']} ({signal['sentiment_score']:.2f})")
        print(f"    Emotion: {signal['emotion_label']} (distress: {signal['is_distress']})")
        print(f"    Distortion: {'Yes' if signal['distortion_indicator'] else 'No'}")
        if signal['distortion_category']:
            print(f"    → Type: {signal['distortion_category']}")
    
    print("\n" + "=" * 70)

await inspect_signals(limit=5)

## Step 5: Run Stage 2 - Behavioral Feature Engineering

This stage aggregates signals and computes risk profiles for each case user.

In [ ]:
from analytics.feature_engineering import BehavioralFeatureEngineer

async def run_stage2(case_users=None, window_days=7):
    """
    Run Stage 2: Feature Engineering and Risk Scoring
    
    Args:
        case_users: List of specific usernames (None = all with signals)
        window_days: Time window for aggregation (default: 7 days)
    """
    print("🚀 Starting Stage 2: Behavioral Feature Engineering")
    print("=" * 70)
    
    engineer = BehavioralFeatureEngineer()
    
    results = await engineer.run_pipeline(
        case_users=case_users,
        window_days=window_days
    )
    
    return results

# Run Stage 2
stage2_results = await run_stage2(window_days=30)

print("\n📊 Stage 2 Results:")
print(f"   • Case users processed: {stage2_results['case_users_processed']}")
print(f"   • Profiles created: {stage2_results['profiles_created']}")
print(f"   • High-risk cases: {len(stage2_results['high_risk_cases'])}")
if stage2_results['high_risk_cases']:
    print(f"   • ⚠️  High-risk users: {', '.join(stage2_results['high_risk_cases'])}")
print(f"   • Duration: {stage2_results['duration_seconds']:.2f}s")

## Step 6: View Risk Profiles

Let's examine the computed risk profiles and see the prioritization.

In [ ]:
async def view_risk_profiles(limit=10):
    """View risk profiles sorted by risk score"""
    db = MongoDB.get_db()
    profiles_collection = db.case_risk_profiles
    
    # Get profiles sorted by risk score (highest first)
    profiles = await profiles_collection.find() \
        .sort('risk_score', -1) \
        .limit(limit) \
        .to_list(length=None)
    
    print("=" * 70)
    print(f"🎯 Top {len(profiles)} Risk Profiles (by Risk Score)")
    print("=" * 70)
    
    for i, profile in enumerate(profiles, 1):
        risk_level = profile['risk_level']
        priority = profile['priority']
        
        # Color coding based on risk
        emoji = "🔴" if risk_level == "High" else "🟡" if risk_level == "Medium" else "🟢"
        
        print(f"\n{emoji} [{i}] {profile['case_user']}")
        print(f"    Risk Score: {profile['risk_score']:.1f}/100")
        print(f"    Level: {risk_level} (Priority {priority})")
        print(f"    Window: {profile['analysis_window_days']} days")
        print(f"    Signals: {profile['total_text_units']} units")
        print(f"    Distortion Rate: {profile['distortion_rate']:.1%}")
        print(f"    Distress Rate: {profile['distress_emotion_rate']:.1%}")
        print(f"    Avg Sentiment: {profile['avg_sentiment_score']:.2f}")
        
        if profile['key_signals']:
            print(f"    Key Signals:")
            for signal in profile['key_signals'][:3]:
                print(f"      • {signal}")
    
    print("\n" + "=" * 70)
    
    return profiles

risk_profiles = await view_risk_profiles(limit=10)

## Step 7: Detailed Case Analysis

Let's do a deep dive into one high-risk case.

In [ ]:
async def analyze_case(username):
    """Detailed analysis of a specific case"""
    db = MongoDB.get_db()
    
    # Get risk profile
    profile = await db.case_risk_profiles.find_one({'case_user': username})
    
    if not profile:
        print(f"❌ No risk profile found for {username}")
        return
    
    # Get signals
    signals = await db.text_units_signals.find({'case_user': username}) \
        .sort('processed_at', -1) \
        .to_list(length=None)
    
    print("=" * 70)
    print(f"📋 Detailed Case Analysis: {username}")
    print("=" * 70)
    
    # Risk overview
    print(f"\n🎯 RISK ASSESSMENT")
    print(f"   Score: {profile['risk_score']:.1f}/100")
    print(f"   Level: {profile['risk_level']}")
    print(f"   Priority: {profile['priority']}")
    
    # Signal breakdown
    print(f"\n📊 SIGNAL BREAKDOWN")
    print(f"   Total text units: {profile['total_text_units']}")
    print(f"   Comments received: {profile['total_comments_received']}")
    print(f"   Captions: {profile['total_captions']}")
    
    # Distortion analysis
    print(f"\n🧠 COGNITIVE DISTORTIONS")
    print(f"   Count: {profile['distortion_count']}")
    print(f"   Rate: {profile['distortion_rate']:.1%}")
    if profile['distortion_categories']:
        print(f"   Categories:")
        for cat, count in profile['distortion_categories'].items():
            print(f"      • {cat}: {count}")
    
    # Sentiment analysis
    print(f"\n😊 SENTIMENT ANALYSIS")
    print(f"   Average: {profile['avg_sentiment_score']:.2f}")
    print(f"   Volatility (σ): {profile['sentiment_std']:.3f}")
    print(f"   Negative rate: {profile['negative_sentiment_rate']:.1%}")
    print(f"   Positive rate: {profile['positive_sentiment_rate']:.1%}")
    
    # Emotion analysis
    print(f"\n😢 EMOTION ANALYSIS")
    print(f"   Distress emotions: {profile['distress_emotion_count']}")
    print(f"   Distress rate: {profile['distress_emotion_rate']:.1%}")
    print(f"   Avg distress score: {profile['avg_distress_score']:.3f}")
    if profile['emotion_distribution']:
        print(f"   Distribution:")
        for emotion, count in sorted(profile['emotion_distribution'].items(), 
                                    key=lambda x: x[1], reverse=True):
            print(f"      • {emotion}: {count}")
    
    # Key signals
    print(f"\n⚠️  KEY SIGNALS")
    for signal in profile['key_signals']:
        print(f"   • {signal}")
    
    # Sample concerning comments
    print(f"\n💬 TOP DISTRESS COMMENTS")
    for i, comment in enumerate(profile['top_distress_comments'][:5], 1):
        print(f"   [{i}] \"{comment}\"")
    
    print("\n" + "=" * 70)

# Analyze the first high-risk user (if any)
if risk_profiles and len(risk_profiles) > 0:
    await analyze_case(risk_profiles[0]['case_user'])
else:
    print("No risk profiles available for detailed analysis")

## Step 8: Export Results for Dashboard

Export the risk profiles in a format ready for dashboard integration.

In [ ]:
import json

async def export_for_dashboard(output_file='dashboard_cases.json'):
    """Export risk profiles in dashboard-ready format"""
    db = MongoDB.get_db()
    
    # Get all profiles sorted by priority
    profiles = await db.case_risk_profiles.find() \
        .sort([('priority', 1), ('risk_score', -1)]) \
        .to_list(length=None)
    
    dashboard_cases = []
    
    for profile in profiles:
        case = {
            'username': profile['case_user'],
            'riskLevel': int(profile['risk_score'] / 20) + 1,  # Convert to 1-5 scale
            'riskScore': round(profile['risk_score'], 1),
            'priority': profile['risk_level'],
            'signals': profile['key_signals'],
            'concerningComments': profile['top_distress_comments'][:3],
            'metrics': {
                'distortionRate': round(profile['distortion_rate'] * 100, 1),
                'distressRate': round(profile['distress_emotion_rate'] * 100, 1),
                'sentimentScore': round(profile['avg_sentiment_score'], 2),
                'volatility': round(profile['sentiment_std'], 2)
            },
            'analysisWindow': f"{profile['analysis_window_days']} days",
            'lastUpdated': profile['last_updated'].isoformat()
        }
        dashboard_cases.append(case)
    
    # Write to JSON file
    with open(output_file, 'w') as f:
        json.dump(dashboard_cases, f, indent=2)
    
    print(f"✅ Exported {len(dashboard_cases)} cases to {output_file}")
    print(f"   • High priority: {sum(1 for c in dashboard_cases if c['priority'] == 'High')}")
    print(f"   • Medium priority: {sum(1 for c in dashboard_cases if c['priority'] == 'Medium')}")
    print(f"   • Low priority: {sum(1 for c in dashboard_cases if c['priority'] == 'Low')}")
    
    return dashboard_cases

exported_cases = await export_for_dashboard()

## Step 9: Pipeline Statistics

View overall analytics statistics.

In [ ]:
async def show_pipeline_stats():
    """Display overall pipeline statistics"""
    db = MongoDB.get_db()
    
    # Count documents
    posts_count = await db.instagram_posts.count_documents({})
    signals_count = await db.text_units_signals.count_documents({})
    profiles_count = await db.case_risk_profiles.count_documents({})
    
    # Risk level breakdown
    high_risk = await db.case_risk_profiles.count_documents({'risk_level': 'High'})
    medium_risk = await db.case_risk_profiles.count_documents({'risk_level': 'Medium'})
    low_risk = await db.case_risk_profiles.count_documents({'risk_level': 'Low'})
    
    # Average risk score
    pipeline = [
        {'$group': {
            '_id': None,
            'avg_risk': {'$avg': '$risk_score'},
            'max_risk': {'$max': '$risk_score'},
            'avg_distortion': {'$avg': '$distortion_rate'},
            'avg_distress': {'$avg': '$distress_emotion_rate'}
        }}
    ]
    stats = await db.case_risk_profiles.aggregate(pipeline).to_list(length=1)
    
    print("=" * 70)
    print("📊 ANALYTICS PIPELINE STATISTICS")
    print("=" * 70)
    
    print(f"\n📦 DATA VOLUMES")
    print(f"   Instagram posts: {posts_count:,}")
    print(f"   NLP signals: {signals_count:,}")
    print(f"   Risk profiles: {profiles_count:,}")
    
    print(f"\n🎯 RISK DISTRIBUTION")
    print(f"   🔴 High risk: {high_risk} ({high_risk/profiles_count*100:.1f}%)")
    print(f"   🟡 Medium risk: {medium_risk} ({medium_risk/profiles_count*100:.1f}%)")
    print(f"   🟢 Low risk: {low_risk} ({low_risk/profiles_count*100:.1f}%)")
    
    if stats:
        s = stats[0]
        print(f"\n📈 AVERAGE METRICS")
        print(f"   Risk score: {s['avg_risk']:.1f}/100")
        print(f"   Max risk score: {s['max_risk']:.1f}/100")
        print(f"   Distortion rate: {s['avg_distortion']:.1%}")
        print(f"   Distress rate: {s['avg_distress']:.1%}")
    
    print("\n" + "=" * 70)

await show_pipeline_stats()

## Summary

This notebook demonstrated the complete analytics pipeline:

1. ✅ **NLP Model Testing** - Verified sentiment, emotion, and distortion detection
2. ✅ **Stage 1 Execution** - Extracted and analyzed text signals
3. ✅ **Stage 2 Execution** - Computed behavioral features and risk scores
4. ✅ **Case Analysis** - Reviewed individual high-risk cases
5. ✅ **Dashboard Export** - Prepared data for visualization

### Next Steps

- **API Integration**: Use the REST API endpoints at `/api/analytics/*`
- **Scheduled Processing**: Set up cron jobs to run the pipeline regularly
- **Dashboard Integration**: Connect the frontend to display risk profiles
- **Alert System**: Implement notifications for high-risk cases
- **Model Tuning**: Adjust thresholds and weights based on validation

### API Endpoints Available

- `POST /api/analytics/extract-signals` - Run Stage 1
- `POST /api/analytics/compute-risk-profiles` - Run Stage 2
- `POST /api/analytics/run-full-pipeline` - Run both stages
- `GET /api/analytics/risk-profiles` - Query risk profiles
- `GET /api/analytics/signals/{username}` - Get user signals
- `GET /api/analytics/stats` - Get overall statistics